# IAC-Flow — ToothFairy3 Left/Right IAC, Colab GPU runner (v1.0)
Two tracks:
1. **nnU-Net v2 baseline** (3-class: bg / Left IAC / Right IAC)
2. **Residual flow refinement** — flow starts from the nnU-Net prediction (x0 = SDF(nnU-Net)),
   trained leakage-free on out-of-fold predictions.

**Upload the whole repo folder** to Colab (or `git clone`), then run the cell below to
`%cd` into it. All paths are repo-root-relative. Full pipeline order is in the README;
this notebook runs the Track A baseline and a Track B smoke test. Runtime → GPU.

## 0. GPU check + Drive mount

In [ ]:
import torch, subprocess
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
from google.colab import drive
drive.mount('/content/drive')


## 1. Point to your data
Upload the **ToothFairy3** folder to Drive (it has `imagesTr/`, `labelsTr/`, `dataset.json`).
Set `TF3_SRC` to that path. Everything else is derived.


In [ ]:
import os
TF3_SRC = "/content/drive/MyDrive/ToothFairy3"          # <-- EDIT to your Drive path
WORK    = "/content/work";  os.makedirs(WORK, exist_ok=True)
os.environ["nnUNet_raw"]          = f"{WORK}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{WORK}/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = f"{WORK}/nnUNet_results"
for d in ["nnUNet_raw","nnUNet_preprocessed","nnUNet_results"]:
    os.makedirs(os.environ[d], exist_ok=True)
assert os.path.isdir(TF3_SRC), f"not found: {TF3_SRC}"
print("source OK:", sorted(os.listdir(TF3_SRC))[:5])


## 2. Install deps

In [ ]:
!pip -q install nnunetv2 nibabel scipy scikit-image


## 3. Build the 3-class dataset (77 -> {bg, L-IAC, R-IAC})
Uses `convert_tf3_to_iac_lr.py`. IAC ids are resolved BY NAME from `dataset.json`
(avoids the mandibular-incisive/lingual canals), then written as Dataset801_IAC_LR.
`--copy-images` because Colab can't symlink into Drive reliably.


In [ ]:
DST = f"{os.environ['nnUNet_raw']}/Dataset801_IAC_LR"
!python data/prepare_iac_dataset.py --src "$TF3_SRC" --dst "$DST" --copy-images --workers 4

---
# TRACK A — nnU-Net v2 baseline

## A1. Plan & preprocess (verifies dataset integrity)

In [ ]:
!nnUNetv2_plan_and_preprocess -d 801 --verify_dataset_integrity


## A2. Train 3D full-res
One fold first (~a few hours on a T4/A100). Colab sessions time out, so
nnU-Net checkpoints to `nnUNet_results` on Drive-backed `WORK` and can resume.


In [ ]:
# Install custom trainers (mirroring OFF for L/R separation) into the nnU-Net package.
import os, shutil, nnunetv2
_da = os.path.join(os.path.dirname(nnunetv2.__file__),
                   "training", "nnUNetTrainer", "variants", "data_augmentation")
shutil.copy("nnunet/nnUNetTrainerIAC.py", _da)
shutil.copy("nnunet/trainer_no_lr_mirror.py", _da)
print("trainers installed ->", _da)

!nnUNetv2_train 801 3d_fullres 0 -tr nnUNetTrainerIAC_NoMirror
# !nnUNetv2_train 801 3d_fullres 1 -tr nnUNetTrainerIAC_NoMirror

## A3. Predict with the baseline

In [ ]:
# put test volumes (named *_0000.nii.gz) in a folder, then:
# !nnUNetv2_predict -i /content/imagesTs -o /content/preds_nnunet \
#     -d 801 -c 3d_fullres -f 0 -tr nnUNetTrainerIAC_NoMirror --disable_tta

# Full residual training. --resume continues from last.pt after a Colab disconnect;
# point --out at Drive so checkpoints survive. Re-run this SAME cell to resume.
# !python flow/train.py --config configs/flow.yaml --fold 0 --resume \
#     --images "$DST/imagesTr" --labels "$DST/labelsTr" \
#     --gt-sdf outputs/gt_sdf --coarse-sdf outputs/coarse_sdf \
#     --out /content/drive/MyDrive/iac_runs/flow_fold0

## B0. Sanity self-test (30s on GPU) — proves the machinery before the long run

In [ ]:
!python flow/selftest.py

## B1. Train the flow model on Dataset801
Patch-based (foreground-oversampled). Adjust `--patch` to fit VRAM
(96 needs ~24 GB; use 64 on a T4). Checkpoints to `runs/fm_iac/best.pt`.


In [ ]:
# Full residual training. --resume continues from last.pt after a Colab disconnect;
# point --out at Drive so checkpoints survive. Re-run this SAME cell to resume.
# !python flow/train.py --config configs/flow.yaml --fold 0 --resume \
#     --images "$DST/imagesTr" --labels "$DST/labelsTr" \
#     --gt-sdf outputs/gt_sdf --coarse-sdf outputs/coarse_sdf \
#     --out /content/drive/MyDrive/iac_runs/flow_fold0

## B2. Sample a mask for one volume

In [ ]:
# Flow inference on a volume uses flow/sliding_window.py (see evaluation/ for scoring).

---
## Notes
- **Baseline vs. extension.** Track A gives a strong, reproducible number to beat.
  Track B is the novel method — compare Dice **and** topological metrics
  (connected components per side should be 1; Betti-0 error; centerline continuity).
- **Why flow matching for the IAC.** Per-voxel classifiers break the thin canal
  where image evidence is weak. A generative sampler over SDFs keeps the
  zero-level-set coherent and can produce multiple plausible masks (uncertainty).
- **SEAL-Flow relationship.** Same flow-matching family, but SEAL-Flow's public
  repo is 2D (PNG cell/nucleus data) with trainer + shape-reg bodies withheld.
  This 3D CBCT formulation is independent and runnable.
